# Tutorial: QSAR Activities and Datasets Smoke Test

Audience:
- AMDockVS developers.

Prerequisites:
- AMDockVS and MolSuite can be imported in the environment.
- The sample library is available at data/Enamine_D10k/Enamine_D10k.sdf.

Learning goals:
- Verify load_activities().
- Verify create_dataset() and split_dataset().
- Verify projection of a dataset to MoleculeSetRef.

## Outline

1. Set up
2. Import a small ligand subset
3. Create a synthetic activity CSV
4. Load activities and create a QSAR dataset
5. Split the roles and project a training set

In [ ]:
from __future__ import annotations

import csv
import sys
import tempfile
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Could not locate repo root")


REPO_ROOT = find_repo_root(Path.cwd())
AMDOCK_SRC = REPO_ROOT / "src"
MOLSUITE_SRC = REPO_ROOT.parent / "MolSuite" / "src"
for path in (AMDOCK_SRC, MOLSUITE_SRC):
    text = str(path)
    if text not in sys.path:
        sys.path.insert(0, text)

from amdockvs.runtime import AMDockVSRuntime

TMP_ROOT = REPO_ROOT / "notebooks" / "_tmp"
TMP_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_DIR = Path(tempfile.mkdtemp(prefix="06_qsar_datasets_", dir=TMP_ROOT))
LIGAND_FILE = REPO_ROOT / "data" / "Enamine_D10k" / "Enamine_D10k.sdf"
assert LIGAND_FILE.exists(), LIGAND_FILE

runtime = AMDockVSRuntime()
runtime.create_project(
    name=f"qsar-datasets-{PROJECT_DIR.name}",
    folder=PROJECT_DIR,
    description="Notebook smoke test for QSAR datasets.",
)

{
    "project_dir": str(PROJECT_DIR),
    "ligand_file": str(LIGAND_FILE),
}


In [ ]:
import_job_ids = runtime.loader.load_ligands(
    [LIGAND_FILE],
    max_records=3,
    executor_name="thread",
)
import_status = runtime.wait_for_jobs(import_job_ids, timeout_s=180.0, poll_s=0.2)
assert all(row.status == "completed" for row in import_status.values()), import_status

ligand_scope = runtime.molecules.select(role="ligand")
ligands = list(runtime.molecules.stream(ligand_scope))
assert len(ligands) == 3, len(ligands)
[(lig.id, lig.name) for lig in ligands]


In [ ]:
activity_csv = PROJECT_DIR / "activities.csv"
with activity_csv.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["name", "IC50"])
    writer.writeheader()
    for index, ligand in enumerate(ligands[:2], start=1):
        writer.writerow({"name": ligand.name, "IC50": float(index) * 10.0})

activity_summary = runtime.qsar.load_activities(
    activity_csv,
    ligand_key="name",
    value_key="IC50",
    endpoint="IC50",
)
assert activity_summary["loaded"] == 2, activity_summary

activities = runtime.qsar.list_activities(endpoint="IC50")
assert len(activities) == 2, activities
activity_summary


In [ ]:
dataset = runtime.qsar.create_dataset(
    name="demo_qsar_dataset",
    molecule_set=runtime.molecules.select(role="ligand"),
    endpoint="IC50",
    require_activity=True,
)
items = runtime.qsar.list_dataset_items(dataset)
assert len(items) == 2, items

split = runtime.qsar.split_dataset(dataset, train=0.5, test=0.5, validation=0.0)
assert split["train"] + split["test"] == 2, split

train_set = runtime.qsar.dataset_molecule_set(dataset, roles=["train"])
train_ids = list(runtime.molecules.stream_ids(runtime.molecules.all(source=train_set)))
assert len(train_ids) == split["train"], (train_ids, split)

{
    "dataset_id": dataset.id,
    "item_count": len(items),
    "split": split,
    "train_set_id": train_set.id,
    "train_ids": train_ids,
}
